# Lab 2 — 도구 사용 (Tool Use)

> **이론 복습 — Session 2 슬라이드**
> - 도구(Tool) = LLM이 호출할 수 있는, 우리가 만든 함수
> - Function calling 4단계: ① 스키마 전달 → ② 모델이 호출 결정 → ③ 우리가 실행 → ④ 결과 반환
> - LLM은 *결정*만 한다. 실행은 *우리 코드*가 한다.

## 학습 목표
1. 분석 도구가 "그냥 Python 함수"임을 확인한다
2. 도구 **JSON 스키마**를 직접 작성한다
3. Function calling **한 바퀴**(①→④)를 손으로 돌려본다
4. 🔧 새 도구를 스키마부터 추가한다


## 0. 준비
- `labs/` 폴더에서 이 노트북을 실행하세요.
- `.env` 에 `GEMINI_API_KEY` 가 있으면 실제 모델, 없으면 `MockLLM` 으로 동작합니다.


In [ ]:
import json
from common.llm import LLMClient
from common import tools

llm = LLMClient()

## 1. 데이터 살펴보기

실습 데이터는 `labs/data/` 에 있고, `common/tools.py` 가 이미 읽어 두었습니다.
모두 `mega-region-ai` 연구의 실제 2024년 통근 데이터에서 추출했습니다.


In [ ]:
print("시군구(regions)      :", len(tools.REGIONS))
print("생활권(living areas) :", len(tools.LIVING_AREAS))
print("통근 통행(OD flows)  :", len(tools.OD_FLOWS))
print()
print("샘플 통행 :", tools.OD_FLOWS[0])
print("샘플 시군구:", tools.REGIONS[0])

## 2. 도구는 "그냥 함수"다

거창한 게 아닙니다. 분석 도구는 입력을 받아 결과를 돌려주는 평범한 함수입니다.
먼저 우리가 직접 불러 봅니다.


In [ ]:
print("get_top_flows(3) — 통행량 상위 3개 구간:")
for f in tools.get_top_flows(3):
    print("  ", f)

print()
print("search_region('강남'):", tools.search_region("강남"))

print()
print("analyze_self_containment() — 자족도가 가장 낮은 3개 생활권:")
for a in tools.analyze_self_containment()[-3:]:
    print("  ", a)

## 3. 도구 스키마 정의 (① 단계)

LLM은 도구의 **코드**를 보지 못합니다. **스키마**(이름·설명·파라미터)만 봅니다.
아래는 도구 3개의 스키마입니다. `description` 은 *모델이 읽는 안내문* 이라는 점에 주목하세요.


In [ ]:
SCHEMA_TOP_FLOWS = {
    "name": "get_top_flows",
    "description": "Return the N largest commute flows between sigungu.",
    "parameters": {
        "type": "object",
        "properties": {
            "n": {"type": "integer", "description": "How many flows to return."},
        },
        "required": [],
    },
}

SCHEMA_SEARCH = {
    "name": "search_region",
    "description": "Find sigungu whose name contains the query text.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {"type": "string", "description": "Part of a region name."},
        },
        "required": ["query"],
    },
}

SCHEMA_SELF_CONTAINMENT = {
    "name": "analyze_self_containment",
    "description": "Self-containment ratio of every living area, highest first.",
    "parameters": {"type": "object", "properties": {}, "required": []},
}

my_tools = [SCHEMA_TOP_FLOWS, SCHEMA_SEARCH, SCHEMA_SELF_CONTAINMENT]
print(f"{len(my_tools)} tool schemas ready.")

## 4. Function calling 한 바퀴

이제 ①스키마를 모델에 주고, ②모델이 어떤 도구를 부를지 보겠습니다.


In [ ]:
question = "통근 통행량이 가장 많은 5개 구간을 알려줘."
messages = [{"role": "user", "content": question}]

reply = llm.generate_with_tools(messages, my_tools)   # (1) + (2)
print("도구를 부르려 하나요?", reply.wants_tool)
for call in reply.tool_calls:
    print("  tool_call:", call.name, call.args)

모델이 일반 텍스트가 아니라 **구조화된 호출**(`tool_call`)로 응답했습니다.
이제 ③ 우리가 실행하고, ④ 결과를 모델에 돌려줍니다.


In [ ]:
# (3) WE execute the tool the model asked for
call = reply.tool_calls[0]
result = tools.call_tool(call.name, call.args)
print("도구 실행 결과:", result)

# (4) send the result back, so the model can write the final answer
messages.append({"role": "assistant", "content": reply.text,
                 "tool_calls": reply.tool_calls})
messages.append({"role": "tool", "name": call.name,
                 "content": json.dumps(result, ensure_ascii=False)})

final = llm.generate_with_tools(messages, my_tools)
print()
print("최종 답변:", final.text)

## 5. 한 함수로 묶기

①~④를 `one_round()` 함수 하나로 묶어 두면 재사용하기 편합니다.
(Session 3에서는 이걸 *여러 번 반복* 하는 루프로 발전시킵니다.)


In [ ]:
def one_round(question, schemas):
    """Run one tool-use round-trip: ask -> (maybe call a tool) -> answer."""
    messages = [{"role": "user", "content": question}]
    reply = llm.generate_with_tools(messages, schemas)

    if not reply.wants_tool:
        return reply.text                       # model answered directly

    call = reply.tool_calls[0]
    result = tools.call_tool(call.name, call.args)
    messages.append({"role": "assistant", "content": reply.text,
                     "tool_calls": reply.tool_calls})
    messages.append({"role": "tool", "name": call.name,
                     "content": json.dumps(result, ensure_ascii=False)})
    return llm.generate_with_tools(messages, schemas).text


print(one_round("강남이 들어가는 지역을 찾아줘.", my_tools))

## 🔧 TODO — 새 도구를 스키마부터 추가하기

`common/tools.py` 에는 `get_inter_area_flows(area_a, area_b)` 함수가 이미 있습니다.
**생활권 간 통행량**을 구하는 도구입니다. 그런데 아직 *스키마* 가 없어 LLM이 쓸 수 없습니다.

아래 `SCHEMA_INTER_AREA` 를 완성하세요 — `area_a`, `area_b` 두 개의 필수 문자열 인자가 필요합니다.
(위 `SCHEMA_SEARCH` 를 본보기로 삼으세요.)


In [ ]:
# 🔧 TODO: 아래 스키마를 완성하세요.
SCHEMA_INTER_AREA = {
    "name": "get_inter_area_flows",
    "description": "",          # TODO: 도구가 하는 일을 영어로 적으세요
    "parameters": {
        "type": "object",
        "properties": {
            # TODO: area_a 와 area_b 를 정의하세요 (둘 다 "string")
        },
        "required": [],         # TODO: 필수 인자 이름을 적으세요
    },
}

# --- test ---
q = "서울 생활권과 경기 남부 생활권 사이의 통근 통행량은?"
print(one_round(q, my_tools + [SCHEMA_INTER_AREA]))

## 정리

- 도구 = 평범한 함수 + 그것을 설명하는 **스키마**
- Function calling 한 바퀴: 스키마 전달 → 호출 결정 → 실행 → 결과 반환
- LLM은 *무엇을 부를지* 만 정하고, 실행은 우리 코드가 한다

**다음 — Session 3**: 이 한 바퀴를 *여러 번 반복* 하는 **에이전트 루프**를 만듭니다.
